In [12]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---------------------------------------------------------
// KERNEL CUDA: Register Tiling + PURE Float4 Vectorization
// ---------------------------------------------------------
__global__ void sgemm_float4_only(int n, const float * __restrict__ A, const float * __restrict__ B, float * __restrict__ C) {

    // NESSUN PADDING. Dimensioni esatte.
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Indice lineare del thread nel blocco (0..255)
    int tid = ty * (BN / TN) + tx;

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    for (int p = 0; p < (n + BK - 1) / BK; ++p) {

        // 1. CARICAMENTO VETTORIZZATO DA GLOBALE A SHARED
        // sA ha 64 righe e 16 colonne (16 float = 4 float4). Totale: 256 float4.
        // I nostri 256 thread caricheranno esattamente un float4 a testa.
        int a_row = tid / 4;          // Riga del tile A (0..63)
        int a_col_v = (tid % 4) * 4;  // Colonna di partenza per il float4 (0, 4, 8, 12)

        int g_a_row = by * BM + a_row;
        int g_a_col = p * BK + a_col_v;

        // Inizializza a zero
        float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
        // Leggi 128 bit (4 float) in una singola istruzione se siamo dentro la matrice
        if (g_a_row < n && g_a_col < n) {
            tmpA = *(const float4*)&A[g_a_row * n + g_a_col];
        }
        // Scrivi 128 bit nella shared in una singola istruzione
        *(float4*)&sA[a_row][a_col_v] = tmpA;


        // sB ha 16 righe e 64 colonne (64 float = 16 float4). Totale: 256 float4.
        // I nostri 256 thread caricheranno esattamente un float4 a testa.
        int b_row = tid / 16;          // Riga del tile B (0..15)
        int b_col_v = (tid % 16) * 4;  // Colonna di partenza per il float4 (0, 4, 8... 60)

        int g_b_row = p * BK + b_row;
        int g_b_col = bx * BN + b_col_v;

        float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
        if (g_b_row < n && g_b_col < n) {
            tmpB = *(const float4*)&B[g_b_row * n + g_b_col];
        }
        *(float4*)&sB[b_row][b_col_v] = tmpB;

        __syncthreads();

        // 2. CALCOLO NEI REGISTRI (Lettura standard, no float4 qui per evitare disallineamenti)
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            #pragma unroll
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];

            #pragma unroll
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            #pragma unroll
            for(int i=0; i<TM; i++) {
                #pragma unroll
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // 3. SCRITTURA FINALE (Qui non usiamo float4 perché le sottomatrici 4x4 non sono sempre contigue in memoria)
    int row_start = by * BM + ty * TM;
    int col_start = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row_start + i;
            int c_col = col_start + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}

// ---------------------------------------------------------
// HOST
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);

    // Questo trucco funziona magicamente bene se N è un multiplo di 4.
    // Per testare performance pure, evitiamo controlli aggiuntivi.

    size_t bytes = (size_t)n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Float4 Pure Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}
'''

with open('my_cuda_float4.cu', 'w') as f:
    f.write(cell_str)

In [13]:
!nvcc -arch=sm_75 -O3 my_cuda_float4.cu -o my_cuda_float4

In [14]:
!nvprof ./my_cuda_float4 15000

==7833== NVPROF is profiling process 7833, command: ./my_cuda_float4 15000
Float4 Pure Time for n=15000: 1.4532 seconds
==7833== Profiling application: ./my_cuda_float4 15000
==7833== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   71.07%  1.45287s         1  1.45287s  1.45287s  1.45287s  sgemm_float4_only(int, float const *, float const *, float*)
                   19.79%  404.69ms         2  202.34ms  201.15ms  203.54ms  [CUDA memcpy HtoD]
                    9.14%  186.85ms         1  186.85ms  186.85ms  186.85ms  [CUDA memcpy DtoH]
      API calls:   64.81%  1.45287s         1  1.45287s  1.45287s  1.45287s  cudaEventSynchronize
                   26.42%  592.34ms         3  197.45ms  187.17ms  203.79ms  cudaMemcpy
                    8.38%  187.93ms         3  62.643ms  149.14us  187.61ms  cudaMalloc
                    0.25%  5.5521ms         3  1.8507ms  934.72us  2.5526ms  cudaFree
                    0.12% 

In [1]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Riportato a 64x64 perché il calcolo degli indici float4 nel kernel è hardcodato per queste dimensioni
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---------------------------------------------------------
// KERNEL CUDA: Register Tiling + PURE Float4 Vectorization
// (IL KERNEL RIMANE INTONSO, PURO E SENZA IF)
// ---------------------------------------------------------
__global__ void sgemm_float4_only(int pad_n, const float * __restrict__ A, const float * __restrict__ B, float * __restrict__ C) {

    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * (BN / TN) + tx;

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    for (int p = 0; p < (pad_n + BK - 1) / BK; ++p) {

        int a_row = tid / 4;
        int a_col_v = (tid % 4) * 4;
        int g_a_row = by * BM + a_row;
        int g_a_col = p * BK + a_col_v;

        float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
        if (g_a_row < pad_n && g_a_col < pad_n) {
            tmpA = *(const float4*)&A[g_a_row * pad_n + g_a_col];
        }
        *(float4*)&sA[a_row][a_col_v] = tmpA;

        int b_row = tid / 16;
        int b_col_v = (tid % 16) * 4;
        int g_b_row = p * BK + b_row;
        int g_b_col = bx * BN + b_col_v;

        float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
        if (g_b_row < pad_n && g_b_col < pad_n) {
            tmpB = *(const float4*)&B[g_b_row * pad_n + g_b_col];
        }
        *(float4*)&sB[b_row][b_col_v] = tmpB;

        __syncthreads();

        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            #pragma unroll
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];

            #pragma unroll
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            #pragma unroll
            for(int i=0; i<TM; i++) {
                #pragma unroll
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    int row_start = by * BM + ty * TM;
    int col_start = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row_start + i;
            int c_col = col_start + j;
            if (c_row < pad_n && c_col < pad_n) {
                C[c_row * pad_n + c_col] = res[i][j];
            }
        }
    }
}

// ---------------------------------------------------------
// HOST: LA MAGIA DEL ZERO-PADDING
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) return 1;

    // N reale richiesto dall'utente (es. 19997)
    int n = atoi(argv[1]);

    // Arrotondiamo al multiplo perfetto del nostro blocco (BM=64)
    // Se n=19997, pad_n diventerà 20032. Perfettamente allineato!
    int pad_n = (n + BM - 1) / BM * BM;

    // Allochiamo la memoria in base alla dimensione maggiorata
    size_t pad_bytes = (size_t)pad_n * pad_n * sizeof(float);

    float *h_a = (float*)malloc(pad_bytes);
    float *h_b = (float*)malloc(pad_bytes);
    float *h_c = (float*)malloc(pad_bytes);

    // Riempimento intelligente
    for (int i = 0; i < pad_n; i++) {
        for (int j = 0; j < pad_n; j++) {
            if (i < n && j < n) {
                // Dentro la matrice reale: dati veri
                h_a[i * pad_n + j] = 2.0f;
                h_b[i * pad_n + j] = 3.0f;
            } else {
                // Fuori dalla matrice reale (la cornice): ZERI SPACCATI
                h_a[i * pad_n + j] = 0.0f;
                h_b[i * pad_n + j] = 0.0f;
            }
            h_c[i * pad_n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, pad_bytes);
    cudaMalloc(&d_b, pad_bytes);
    cudaMalloc(&d_c, pad_bytes);

    cudaMemcpy(d_a, h_a, pad_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, pad_bytes, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((pad_n + BN - 1) / BN, (pad_n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    // ATTENZIONE: Passiamo al Kernel pad_n, non n. La GPU è ignara dell'inganno!
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(pad_n, d_a, d_b, d_c);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // Mostriamo all'utente il tempo speso per la matrice "padded"
    printf("Float4 Time for REAL_N=%d (Padded to %d): %.4f seconds\n", n, pad_n, milliseconds / 1000.0f);

    cudaMemcpy(h_c, d_c, pad_bytes, cudaMemcpyDeviceToHost);

    // Quando salviamo il file, estraiamo solo il nucleo nxn reale!
    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                // Leggiamo usando la larghezza della matrice allocata (pad_n)
                fprintf(f, "%.0f ", h_c[i * pad_n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}
'''

with open('my_cuda_float4_adv.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 my_cuda_float4_adv.cu -o my_cuda_float4_adv

In [5]:
!nvprof ./my_cuda_float4_adv 20033

==2329== NVPROF is profiling process 2329, command: ./my_cuda_float4_adv 20033
Float4 Time for REAL_N=20033 (Padded to 20096): 3.2362 seconds
==2329== Profiling application: ./my_cuda_float4_adv 20033
==2329== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   75.05%  3.23589s         1  3.23589s  3.23589s  3.23589s  sgemm_float4_only(int, float const *, float const *, float*)
                   16.24%  700.38ms         2  350.19ms  349.35ms  351.03ms  [CUDA memcpy HtoD]
                    8.70%  375.20ms         1  375.20ms  375.20ms  375.20ms  [CUDA memcpy DtoH]
      API calls:   71.82%  3.23590s         1  3.23590s  3.23590s  3.23590s  cudaEventSynchronize
                   23.89%  1.07643s         3  358.81ms  349.59ms  375.57ms  cudaMemcpy
                    4.07%  183.55ms         3  61.185ms  152.03us  183.14ms  cudaMalloc
                    0.15%  6.7323ms         3  2.2441ms  1.7538ms  2.8740ms  cudaFree
